# HOMER × ENIGMA cross-disorder spatial validation

The ENIGMA consortium publishes meta-analyses of brain abnormalities across psychiatric disorders. Each working group (ASD, schizophrenia, MDD, bipolar, ADHD, OCD, 22q11.2) ships per-Desikan-Killiany-region Cohen's d effect-size maps showing where each disorder concentrates anatomically.

**Two-phase test:**

- **Phase 1 (in-sandbox)**: generate HOMER's per-disorder predicted human spatial patterns from MOESM4 (autism) + MOESM5 (other psych conditions) gene sets routed through π. Compute cross-disorder correlation matrix at parcel resolution.

- **Phase 2 (external data)**: compare against ENIGMA's published per-disorder Cohen's d maps. Test whether HOMER's predictions are disorder-specific (matched-disorder correlations beat mismatched) and how strongly HOMER's generic brain-disorder geometry aligns with ENIGMA's observed psychiatric anatomy.


## Setup

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

from homer.data import load_cached

pi = np.load(ROOT / 'outputs/coupling/pi_fc_plus_SC_with_all_packs.npy')
H, _ = load_cached('human', cache_dir=str(ROOT / 'outputs/anndata'))
print(f'HOMER π: {pi.shape}')


## Phase 1 — HOMER per-disorder predictions at parcel resolution

For each disorder (autism, bipolar, schizophrenia, ADHD), translate its gene set through π → 2,094-parcel predicted human spatial pattern. The question: are these patterns *disorder-specific* or all the *same generic brain-disorder geometry*?


In [ ]:
result = json.loads((ROOT / 'outputs/logs/enigma_phase1_per_disorder.json').read_text())
preds = np.load(ROOT / 'outputs/coupling/per_disorder_predictions.npz')
disorders = result['disorders']
corr_mat = np.array(result['correlation_matrix'])

print(f"{'Disorder pair':<40s} | Pearson r")
print('-' * 60)
for i, di in enumerate(disorders):
    for j, dj in enumerate(disorders):
        if i < j:
            print(f"  {di:<18s} ↔ {dj:<18s} | {corr_mat[i,j]:+.4f}")

iu = np.triu_indices(len(disorders), k=1)
mean_off = float(corr_mat[iu].mean())
print(f'\nMean off-diagonal r: {mean_off:+.4f}')
print('Interpretation:')
print('  r ≈ 1.0 means HOMER predicts essentially the same spatial pattern for all disorders.')
print('  r ≈ 0 would mean each disorder has a distinctive predicted pattern.')


**Result: HOMER's predictions are NOT disorder-specific** — r > 0.97 across all disorder pairs at parcel resolution. The mean off-diagonal r is +0.988, almost identical predictions.

This sharpens our earlier finding (Pagani cross-disease specificity at 8-network resolution gave r ≈ +0.4 across disorders with overlapping CIs). At full parcel resolution, the *differences* between disorders' predictions are essentially noise — HOMER produces a single "generic brain-disorder spatial geometry" regardless of which disorder's gene set goes in.

**What this means:** HOMER's gene-spatial translation captures shared psychiatric-perturbation geometry across species, but does NOT distinguish autism from schizophrenia from bipolar from ADHD. The Test 3 result (autism r=+0.43 against Pagani's observed Δ) is the same correlation any psychiatric gene set would produce.


## Is the shared geometry just gene-set overlap?

The disorder gene sets overlap heavily (the non-autism sets are essentially *nested* in the 1,713-gene autism set), so identical inputs could trivially give identical outputs. The disorder-unique test controls for this: route only the genes in disorder A-not-B vs B-not-A (`experiments/enigma_cross_disorder/04_disorder_unique.py`).

In [ ]:
du = json.loads((ROOT / 'outputs/logs/enigma_disorder_unique.json').read_text())
print(f"full-set off-diagonal r: {du['full_offdiag_mean']:+.3f}")
print('pairwise relative-unique (disjoint gene sets):')
for pair, v in du['pairwise_relative_unique'].items():
    print(f"  {pair:<34} A-only={v['n_A_only']:>4} B-only={v['n_B_only']:>4}  r={v['r']:+.3f}")

**Robust, not an overlap artefact.** Even *fully disjoint* gene sets (e.g. bipolar-only vs schizophrenia-only) route to near-identical human maps (r ≈ +0.98). So HOMER carries a genuine **shared psychiatric spatial geometry**, not a trivial consequence of the gene sets overlapping.

## Visualise Phase 1

In [ ]:
from IPython.display import Image, display
fig_path = ROOT / 'outputs/figures/enigma_phase1_per_disorder.png'
if fig_path.exists():
    display(Image(str(fig_path)))


## Phase 2 — comparison against ENIGMA observed disease maps (now run)

Does HOMER's shared geometry match the real transdiagnostic cortical signature? We compare HOMER's generic predicted map to ENIGMA observed cortical-thickness Cohen's d (Desikan-Killiany; ENIGMA Toolbox `summary_statistics`, staged in `data_external/enigma/`), including a transdiagnostic average across ASD/SCZ/BD/ADHD and a spin null over DK centroids (`05_transdiagnostic.py`).

In [ ]:
td = json.loads((ROOT / 'outputs/logs/enigma_transdiagnostic.json').read_text())['results']
ta = td['transdiagnostic_average']
print(f"HOMER generic vs ENIGMA transdiagnostic average: r={ta['pearson_r']:+.3f} "
      f"spin p={ta['spin_p']:.3f}")
for k in ['asd','schizophrenia','bipolar','adhd','mdd','ocd']:
    if k in td:
        tag = '' if td[k].get('in_gene_sets', True) else '(held-out)'
        print(f"  {k:<14} r={td[k]['pearson_r']:+.3f}  spin p={td[k]['spin_p']:.3f}  {tag}")

**Honest negative against external maps.** HOMER's generic map does *not* align with the observed ENIGMA cortical-thickness signature beyond spatial autocorrelation (transdiagnostic r = −0.27, spin p = 0.24; n.s. for every disorder and for held-out MDD/OCD). So the shared geometry is a property of the gene→π routing, **not** validated against observed psychiatric atrophy. Reporting both — the robust internal shared-geometry result and this external null — is what keeps the claim honest.

## Where this fits in the HOMER × literature picture

| Test | Granularity | Result | Verdict |
|---|---|---|---|
| Pagani Test 2c | Network-pair Δ | r=+0.547, p=0.0006 | **Strong** |
| Pagani cross-disease (8 nets) | 4 disorders' gene sets | r≈+0.4 each, no specificity | Caveat |
| **ENIGMA Phase 1** | **2,094 parcels per disorder** | **Off-diag r=+0.988** | **Confirms no disorder-specificity** |
| ENIGMA Phase 2 (pending) | DK regions, HOMER vs observed | Awaiting data | TBD |

The Phase 1 result is informative as-is. It establishes that HOMER's gene-spatial translation has *no disorder specificity at all* at parcel resolution. Whatever HOMER captures cross-species, it's a generic psychiatric-perturbation geometry — useful for understanding shared biology, but not a diagnostic distinguisher.

This is the cleanest possible statement of HOMER's limit in the gene-spatial domain: HOMER captures the spatial structure that psychiatric brain disorders share, not what makes each one unique.

For full details: [`docs/03_results.md`](../docs/03_results.md). Code: `experiments/enigma_cross_disorder/`.
